# Game Player Performance Prediction Using Machine Learning
## Free Fire / Battle Royale themed MCA Data Science project

**Goal:** train a Random Forest classifier to categorize a match as Low, Medium, or High performance, then evaluate it and prepare the Streamlit dashboard.

> **Data integrity note:** the included CSV is synthetic demonstration data, not official Free Fire data. Metrics are for this synthetic dataset only.

## 1. Install and import libraries

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn streamlit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import files


## 2. Upload the dataset

In [ ]:
# Upload battle_royale_synthetic_matches.csv from the project ZIP's data folder.
uploaded = files.upload()
csv_name = next((name for name in uploaded if name.lower().endswith('.csv')), None)
if csv_name is None:
    raise ValueError("Please upload the CSV file.")
df = pd.read_csv(csv_name)
df.head()


## 3. Explore and inspect data

In [ ]:
print("Shape:", df.shape)
print("\\nMissing values:\\n", df.isna().sum())
print("\\nTarget distribution:\\n", df["performance_tier"].value_counts())
display(df.describe())


In [ ]:
df["performance_tier"].value_counts().plot(kind="bar", title="Performance tier distribution")
plt.xlabel("Tier"); plt.ylabel("Number of matches"); plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="kills", y="damage", hue="performance_tier", alpha=.6)
plt.title("Kills vs Damage"); plt.show()


## 4. Prepare features and target

In [ ]:
features = [
    "kills", "assists", "damage", "survival_minutes", "placement",
    "accuracy_percent", "headshots", "revives", "match_duration_minutes"
]
X = df[features].copy()
y = df["performance_tier"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))


## 5. Train Random Forest model

In [ ]:
model = RandomForestClassifier(
    n_estimators=250, max_depth=10, min_samples_leaf=3,
    class_weight="balanced", random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Hold-out accuracy on synthetic data: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, zero_division=0))


## 6. Confusion matrix and feature importance

In [ ]:
labels = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix")
plt.show()

importance = pd.Series(model.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", title="Random Forest feature importance")
plt.xlabel("Importance"); plt.show()


## 7. Predict a sample player

In [ ]:
sample = pd.DataFrame([{
    "kills": 6, "assists": 2, "damage": 520, "survival_minutes": 17,
    "placement": 5, "accuracy_percent": 38, "headshots": 2,
    "revives": 1, "match_duration_minutes": 22
}])[features]

prediction = model.predict(sample)[0]
probabilities = dict(zip(model.classes_, model.predict_proba(sample)[0].round(3)))
print("Predicted tier:", prediction)
print("Model class probabilities:", probabilities)


## 8. Download the Streamlit app and run it

In [ ]:
# Upload app.py from the extracted project folder when prompted.
uploaded_app = files.upload()
app_name = next((name for name in uploaded_app if name == "app.py"), None)
if app_name:
    print("app.py uploaded. To run locally, use: streamlit run app.py")
else:
    print("Upload app.py to continue.")


### Optional Streamlit preview in Colab
If you upload both `app.py` and the CSV into the same Colab working directory, try:
```python
!mkdir -p data
!cp battle_royale_synthetic_matches.csv data/
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit.log &
!npx localtunnel --port 8501
```
The tunnel prints a temporary URL. Public tunnel services can be unreliable; local execution is the recommended stable option.

### Conclusion
This notebook demonstrates data exploration, supervised classification, model evaluation, and an interactive app workflow. Replace synthetic data with a documented, appropriately licensed real dataset before making real-world claims.